# CEREBRO — Step 09: Adaptive Multimodal Extraction

**Experiment:** EXP-EXTRACT-001  
**Stage:** Adaptive Ingestion — Multimodal Extraction  
**Status:** Experimental  
**Depends On:** Step 07 Artifact Identification + Step 08 Capability Routing

---

## 1. Purpose

Step 09 evaluates whether CEREBRO can execute the appropriate extraction capability for an identified artifact without hardcoding the ingestion pipeline around a single modality.

Previous experiments established:

**Step 07**

Unknown Artifact → Deterministic Inspection → Physical Format → Logical Artifact Type

**Step 08**

Artifact Type → Required Processing Capability → Processing Route

Step 09 now executes that route.

The objective is to transform heterogeneous source artifacts into provenance-aware derived representations while preserving the original artifact and its source structure.

---

## 2. Research Question

Can CEREBRO process different artifact modalities using the appropriate extraction capability while producing a consistent downstream representation with complete provenance?

The experiment should answer:

1. Was the correct extraction capability invoked?
2. Was the lightest appropriate implementation used?
3. What information was successfully extracted?
4. What information could not be extracted?
5. Was source structure/location preserved?
6. Was provenance preserved?
7. Was AI required?
8. Is the extraction output suitable for quality evaluation?

---

## 3. Core Principle

> Use the right capability, implemented by the lightest appropriate tool or model.

CEREBRO should not default to an LLM for artifact extraction.

Examples:

TXT
→ deterministic UTF-8 extraction

DOCX
→ OOXML/document parser

PPTX
→ OOXML/presentation parser

XLSX
→ spreadsheet parser

Native PDF
→ PDF text/layout parser

Scanned PDF
→ OCR

Image containing text
→ OCR

Visual image
→ vision processing

Audio
→ ASR

Video
→ audio + temporal + visual processing

AI should be introduced only when deterministic processing cannot sufficiently interpret the source.

---

## 4. Architectural Boundary

Step 09 performs **extraction**, not knowledge construction.

The following remain separate:

**Artifact**
↓
Identification
↓
Capability Routing
↓
**Extraction ← Step 09**
↓
Quality Evaluation
↓
Normalization
↓
Human Review
↓
Registration
↓
Knowledge Construction

Step 09 must NOT:

- register the artifact,
- create trusted Knowledge Fragments,
- generate trusted relationships,
- modify the Knowledge Galaxy,
- modify the Knowledge Timeline,
- silently enrich extracted content,
- invent missing metadata,
- promote AI output to trusted knowledge.

---

## 5. Input Contract

Step 09 receives:

### Original Artifact

- filename
- raw bytes
- SHA-256

### Identification

- artifact type
- modality
- container type where applicable
- identification evidence
- identification method

### Capability Routing

- required capability
- processing strategy
- selected implementation where available
- fallback/escalation policy

The extraction layer should not independently reinterpret the artifact unless contradictory evidence is discovered.

---

## 6. Supported Extraction Paths

The experimental architecture should support the following logical routes.

| Artifact | Primary Capability | Initial Processing |
|---|---|---|
| TXT | TEXT_EXTRACTION | UTF-8 parser |
| PDF | PDF_EXTRACTION | native parser / OCR decision |
| DOCX | DOCUMENT_EXTRACTION | OOXML parser |
| PPTX | PRESENTATION_EXTRACTION | OOXML parser |
| XLSX | SPREADSHEET_EXTRACTION | workbook parser |
| PNG/JPEG | IMAGE_EXTRACTION | OCR / vision decision |
| WAV/MP3 | AUDIO_TRANSCRIPTION | ASR |
| MP4 | VIDEO_EXTRACTION | ASR + keyframes + vision |

These are capability routes rather than vendor-specific implementations.

---

## 7. Multimodal Principle

CEREBRO should not reduce every artifact immediately to plain text.

Instead:

**Original Artifact**
↓
**Modality-Specific Derived Representation**
↓
**Normalized Knowledge Representation**

For example:

### Presentation

PPTX
├── Slide 1
│   ├── Text
│   ├── Images
│   ├── Tables
│   ├── Notes
│   └── Relationships
├── Slide 2
└── Slide N

### Image

Image
├── OCR Text
├── Visual Objects
├── Description
├── Entities
└── Spatial Information

### Audio

Audio
├── Transcript
├── Segments
├── Timestamps
└── Speaker Information where available

### Video

Video
├── Audio Transcript
├── Transcript Timestamps
├── Keyframes
├── OCR
├── Visual Information
└── Temporal Relationships

The original modality must remain available.

---

## 8. Canonical Derived Representation

Step 09 should begin converging on a modality-independent extraction contract.

Conceptually:

    ExtractedArtifact
    ├── artifact_id / temporary artifact reference
    ├── filename
    ├── sha256
    ├── artifact_type
    ├── modality
    │
    ├── representations[]
    │   ├── representation_type
    │   ├── content
    │   ├── source_location
    │   ├── extraction_method
    │   ├── tool
    │   ├── model
    │   └── provenance
    │
    ├── extraction_summary
    │
    └── integrity

Downstream CEREBRO components should consume this contract rather than depending directly on TXT, PDF, PPTX, Whisper, OCR, or a particular vendor.

---

## 9. Source Location

Source location is critical to CEREBRO.

Different modalities require different location models.

### TXT

    character_range:
        start_char
        end_char

### PDF

    page:
        page_number

Optional future extension:

    bounding_box

### PPTX

    slide:
        slide_number

Optional:

    object_id
    shape_index

### DOCX

    paragraph:
        paragraph_number

Optional:

    section
    table
    object

### Image

    bounding_box:
        x
        y
        width
        height

### Audio

    time_range:
        start_seconds
        end_seconds

### Video

    time_range:
        start_seconds
        end_seconds

plus potentially:

    frame_number

The location model must allow future Assisted Recollection to navigate back to the evidence.

---

## 10. Provenance Requirements

Every derived representation must retain:

- source artifact identity,
- source SHA-256,
- source location,
- extraction capability,
- extraction implementation,
- extraction method,
- tool/model where applicable,
- AI usage,
- timestamp,
- transformation relationship.

Conceptually:

**Source Artifact**
↓
`DERIVED_FROM`
↓
**Extracted Representation**

No extracted representation should become detached from its source.

---

## 11. Deterministic vs AI Extraction

Extraction methods should explicitly declare their nature.

Examples:

    method_type = deterministic

or:

    method_type = model_based

or:

    method_type = hybrid

This distinction should remain visible throughout the integrity chain.

Example:

    PPTX slide text
        method_type = deterministic
        method = OOXML extraction

    PPTX diagram interpretation
        method_type = model_based
        method = vision analysis

These outputs should not be treated as equally authoritative.

---

## 12. Extraction State

Each derived representation should have an explicit state.

Suggested experimental states:

- EXTRACTED
- PARTIALLY_EXTRACTED
- EXTRACTION_FAILED
- REQUIRES_SPECIALIST_PROCESSING
- REQUIRES_AI
- REQUIRES_HUMAN_REVIEW

This prevents partial extraction from silently appearing complete.

---

## 13. AI Routing

If model-based processing becomes necessary:

**Required Capability**
↓
Can deterministic processing satisfy it?
↓
YES → deterministic processing
↓
NO
↓
Can local model satisfy it?
↓
YES → local model
↓
Evaluate
↓
Insufficient?
↓
Frontier escalation if permitted

The routing decision should consider:

- privacy,
- task complexity,
- extraction quality,
- confidence/evidence,
- cost,
- latency,
- available local capability.

The model itself should remain behind a CEREBRO adapter.

---

## 14. Extraction Quality Boundary

Step 09 records extraction evidence but does not yet claim extraction quality.

For example:

Step 09 may record:

    extracted_characters = 3,521
    slides_processed = 12
    images_detected = 7
    tables_detected = 2

But should not automatically claim:

    extraction_accuracy = 95%

unless suitable ground truth exists.

Formal quality evaluation belongs to Step 10.

---

## 15. Failure Handling

Extraction must fail explicitly.

Examples:

### Unsupported format

    status = UNSUPPORTED

### Corrupted artifact

    status = EXTRACTION_FAILED

### Partial extraction

    status = PARTIALLY_EXTRACTED

### Capability unavailable

    status = CAPABILITY_UNAVAILABLE

### Requires escalation

    status = ESCALATION_REQUIRED

CEREBRO must not silently replace failed extraction with fabricated content.

---

## 16. Experimental Method

### Phase A — Load Previous Experiment

Load the persisted artifact-identification and capability-routing records.

Verify:

- artifact SHA-256,
- artifact type,
- modality,
- required capability.

---

### Phase B — Validate Original Artifact

Recalculate SHA-256 from the raw bytes.

The calculated checksum must match the checksum used during identification.

If not:

**STOP — SOURCE INTEGRITY FAILURE**

---

### Phase C — Select Extraction Adapter

Use the capability-routing decision to select the appropriate extraction adapter.

Example:

    PRESENTATION_EXTRACTION
              ↓
        PPTX adapter

The notebook should not choose the adapter directly from the filename.

---

### Phase D — Execute Extraction

Run the selected capability.

Produce one or more derived representations.

---

### Phase E — Preserve Source Locations

Every extracted unit must identify where it originated.

Examples:

    Slide 3
    Page 7
    Paragraph 12
    01:34–01:42
    Bounding box
    Character 120–187

---

### Phase F — Produce Extraction Manifest

Generate a structured manifest describing:

- what was processed,
- how it was processed,
- what was extracted,
- what was not extracted,
- provenance,
- source locations,
- processing status.

---

### Phase G — Integrity Validation

Validate:

- original checksum preserved,
- every representation has provenance,
- every representation has source location where applicable,
- deterministic/model distinction retained,
- no knowledge registration occurred.

---

## 17. Success Criteria

EXP-EXTRACT-001 succeeds when:

1. The artifact is processed using its routed capability.
2. Original bytes remain unchanged.
3. Original SHA-256 remains unchanged.
4. Extraction method is recorded.
5. Tool/model information is recorded where applicable.
6. Extracted representations retain source location.
7. Extracted representations retain source provenance.
8. Partial extraction is explicitly represented.
9. AI usage is explicitly represented.
10. No artifact is registered automatically.
11. No trusted Knowledge Fragment is created.
12. Output can be consumed by Step 10 quality evaluation.

---

## 18. Failure Criteria

The experiment fails if:

- processor is selected solely from filename extension,
- original artifact is modified,
- provenance is lost,
- source location is lost where technically available,
- model-generated content is represented as deterministic extraction,
- partial extraction is reported as complete,
- missing content is fabricated,
- artifact is registered automatically,
- Knowledge Fragments are created prematurely,
- extraction implementation leaks into CEREBRO Core contracts.

---

## 19. Expected Output

The experiment should produce:

    extraction_result
    │
    ├── experiment_id
    ├── source
    │   ├── filename
    │   ├── sha256
    │   ├── artifact_type
    │   └── modality
    │
    ├── routing
    │   ├── capability
    │   ├── implementation
    │   └── method_type
    │
    ├── representations[]
    │   ├── representation_id
    │   ├── representation_type
    │   ├── content
    │   ├── source_location
    │   ├── extraction_method
    │   └── provenance
    │
    ├── extraction_summary
    │
    ├── integrity
    │
    └── status

---

## 20. Output Persistence

Persist the experimental result independently from the trusted CEREBRO knowledge store.

Suggested location:

    poc/outputs/experiments/
        EXP-EXTRACT-001.json

Large derived artifacts should be stored separately where appropriate.

Example:

    poc/data/derived/
        <source-sha256>/
            text/
            images/
            audio/
            frames/

The experiment manifest should reference these representations rather than embedding large binary payloads directly.

---

## 21. Relationship to CEREBRO Integrity Framework

Step 09 extends the CEREBRO integrity chain:

**Original Artifact**
↓
**Identification**
↓
**Capability Routing**
↓
**Extraction**
↓
**Derived Representation**

Later:

**Derived Representation**
↓
Quality Evaluation
↓
Normalization
↓
Human Review
↓
Registration
↓
Knowledge Construction
↓
Retrieval
↓
Assisted Recollection
↓
Original Evidence

The original artifact remains the root of provenance.

---

## 22. Expected Architectural Learning

Step 09 should help answer whether CEREBRO can maintain a common ingestion architecture while allowing different modalities to use different processing capabilities.

The desired result is not:

    TXT pipeline
    PDF pipeline
    PPTX pipeline
    Audio pipeline
    Video pipeline

as unrelated systems.

Instead:

                    ARTIFACT
                       │
                       ▼
                   IDENTIFY
                       │
                       ▼
                    ROUTE
                       │
          ┌────────────┼────────────┐
          ▼            ▼            ▼
     deterministic   local AI    specialist
          │            │            │
          └────────────┼────────────┘
                       ▼
             DERIVED REPRESENTATION
                       │
                       ▼
                  EVALUATE
                       │
                       ▼
                  NORMALIZE

The modality-specific implementation should remain behind the extraction capability boundary.

---

## 23. Experiment Principle

> **Different artifacts require different ways of being understood, but they should enter the Digital Knowledge Twin through the same integrity framework.**

Step 09 therefore tests CEREBRO's ability to move from:

**“I know what this artifact is.”**

to:

**“I can extract what it contains while preserving exactly where that information came from.”**

# CEREBRO — Step 09: Adaptive Multimodal Extraction

**Experiment:** EXP-EXTRACT-001  
**Stage:** Adaptive Ingestion — Multimodal Extraction  
**Status:** Experimental  
**Depends On:** Step 07 Artifact Identification + Step 08 Capability Routing

---

## 1. Purpose

Step 09 evaluates whether CEREBRO can execute the appropriate extraction capability for an identified artifact without hardcoding the ingestion pipeline around a single modality.

Previous experiments established:

**Step 07**

Unknown Artifact → Deterministic Inspection → Physical Format → Logical Artifact Type

**Step 08**

Artifact Type → Required Processing Capability → Processing Route

Step 09 now executes that route.

The objective is to transform heterogeneous source artifacts into provenance-aware derived representations while preserving the original artifact and its source structure.

---

## 2. Research Question

Can CEREBRO process different artifact modalities using the appropriate extraction capability while producing a consistent downstream representation with complete provenance?

The experiment should answer:

1. Was the correct extraction capability invoked?
2. Was the lightest appropriate implementation used?
3. What information was successfully extracted?
4. What information could not be extracted?
5. Was source structure/location preserved?
6. Was provenance preserved?
7. Was AI required?
8. Is the extraction output suitable for quality evaluation?

---

## 3. Core Principle

> Use the right capability, implemented by the lightest appropriate tool or model.

CEREBRO should not default to an LLM for artifact extraction.

Examples:

TXT
→ deterministic UTF-8 extraction

DOCX
→ OOXML/document parser

PPTX
→ OOXML/presentation parser

XLSX
→ spreadsheet parser

Native PDF
→ PDF text/layout parser

Scanned PDF
→ OCR

Image containing text
→ OCR

Visual image
→ vision processing

Audio
→ ASR

Video
→ audio + temporal + visual processing

AI should be introduced only when deterministic processing cannot sufficiently interpret the source.

---

## 4. Architectural Boundary

Step 09 performs **extraction**, not knowledge construction.

The following remain separate:

**Artifact**
↓
Identification
↓
Capability Routing
↓
**Extraction ← Step 09**
↓
Quality Evaluation
↓
Normalization
↓
Human Review
↓
Registration
↓
Knowledge Construction

Step 09 must NOT:

- register the artifact,
- create trusted Knowledge Fragments,
- generate trusted relationships,
- modify the Knowledge Galaxy,
- modify the Knowledge Timeline,
- silently enrich extracted content,
- invent missing metadata,
- promote AI output to trusted knowledge.

---

## 5. Input Contract

Step 09 receives:

### Original Artifact

- filename
- raw bytes
- SHA-256

### Identification

- artifact type
- modality
- container type where applicable
- identification evidence
- identification method

### Capability Routing

- required capability
- processing strategy
- selected implementation where available
- fallback/escalation policy

The extraction layer should not independently reinterpret the artifact unless contradictory evidence is discovered.

---

## 6. Supported Extraction Paths

The experimental architecture should support the following logical routes.

| Artifact | Primary Capability | Initial Processing |
|---|---|---|
| TXT | TEXT_EXTRACTION | UTF-8 parser |
| PDF | PDF_EXTRACTION | native parser / OCR decision |
| DOCX | DOCUMENT_EXTRACTION | OOXML parser |
| PPTX | PRESENTATION_EXTRACTION | OOXML parser |
| XLSX | SPREADSHEET_EXTRACTION | workbook parser |
| PNG/JPEG | IMAGE_EXTRACTION | OCR / vision decision |
| WAV/MP3 | AUDIO_TRANSCRIPTION | ASR |
| MP4 | VIDEO_EXTRACTION | ASR + keyframes + vision |

These are capability routes rather than vendor-specific implementations.

---

## 7. Multimodal Principle

CEREBRO should not reduce every artifact immediately to plain text.

Instead:

**Original Artifact**
↓
**Modality-Specific Derived Representation**
↓
**Normalized Knowledge Representation**

For example:

### Presentation

PPTX
├── Slide 1
│   ├── Text
│   ├── Images
│   ├── Tables
│   ├── Notes
│   └── Relationships
├── Slide 2
└── Slide N

### Image

Image
├── OCR Text
├── Visual Objects
├── Description
├── Entities
└── Spatial Information

### Audio

Audio
├── Transcript
├── Segments
├── Timestamps
└── Speaker Information where available

### Video

Video
├── Audio Transcript
├── Transcript Timestamps
├── Keyframes
├── OCR
├── Visual Information
└── Temporal Relationships

The original modality must remain available.

---

## 8. Canonical Derived Representation

Step 09 should begin converging on a modality-independent extraction contract.

Conceptually:

    ExtractedArtifact
    ├── artifact_id / temporary artifact reference
    ├── filename
    ├── sha256
    ├── artifact_type
    ├── modality
    │
    ├── representations[]
    │   ├── representation_type
    │   ├── content
    │   ├── source_location
    │   ├── extraction_method
    │   ├── tool
    │   ├── model
    │   └── provenance
    │
    ├── extraction_summary
    │
    └── integrity

Downstream CEREBRO components should consume this contract rather than depending directly on TXT, PDF, PPTX, Whisper, OCR, or a particular vendor.

---

## 9. Source Location

Source location is critical to CEREBRO.

Different modalities require different location models.

### TXT

    character_range:
        start_char
        end_char

### PDF

    page:
        page_number

Optional future extension:

    bounding_box

### PPTX

    slide:
        slide_number

Optional:

    object_id
    shape_index

### DOCX

    paragraph:
        paragraph_number

Optional:

    section
    table
    object

### Image

    bounding_box:
        x
        y
        width
        height

### Audio

    time_range:
        start_seconds
        end_seconds

### Video

    time_range:
        start_seconds
        end_seconds

plus potentially:

    frame_number

The location model must allow future Assisted Recollection to navigate back to the evidence.

---

## 10. Provenance Requirements

Every derived representation must retain:

- source artifact identity,
- source SHA-256,
- source location,
- extraction capability,
- extraction implementation,
- extraction method,
- tool/model where applicable,
- AI usage,
- timestamp,
- transformation relationship.

Conceptually:

**Source Artifact**
↓
`DERIVED_FROM`
↓
**Extracted Representation**

No extracted representation should become detached from its source.

---

## 11. Deterministic vs AI Extraction

Extraction methods should explicitly declare their nature.

Examples:

    method_type = deterministic

or:

    method_type = model_based

or:

    method_type = hybrid

This distinction should remain visible throughout the integrity chain.

Example:

    PPTX slide text
        method_type = deterministic
        method = OOXML extraction

    PPTX diagram interpretation
        method_type = model_based
        method = vision analysis

These outputs should not be treated as equally authoritative.

---

## 12. Extraction State

Each derived representation should have an explicit state.

Suggested experimental states:

- EXTRACTED
- PARTIALLY_EXTRACTED
- EXTRACTION_FAILED
- REQUIRES_SPECIALIST_PROCESSING
- REQUIRES_AI
- REQUIRES_HUMAN_REVIEW

This prevents partial extraction from silently appearing complete.

---

## 13. AI Routing

If model-based processing becomes necessary:

**Required Capability**
↓
Can deterministic processing satisfy it?
↓
YES → deterministic processing
↓
NO
↓
Can local model satisfy it?
↓
YES → local model
↓
Evaluate
↓
Insufficient?
↓
Frontier escalation if permitted

The routing decision should consider:

- privacy,
- task complexity,
- extraction quality,
- confidence/evidence,
- cost,
- latency,
- available local capability.

The model itself should remain behind a CEREBRO adapter.

---

## 14. Extraction Quality Boundary

Step 09 records extraction evidence but does not yet claim extraction quality.

For example:

Step 09 may record:

    extracted_characters = 3,521
    slides_processed = 12
    images_detected = 7
    tables_detected = 2

But should not automatically claim:

    extraction_accuracy = 95%

unless suitable ground truth exists.

Formal quality evaluation belongs to Step 10.

---

## 15. Failure Handling

Extraction must fail explicitly.

Examples:

### Unsupported format

    status = UNSUPPORTED

### Corrupted artifact

    status = EXTRACTION_FAILED

### Partial extraction

    status = PARTIALLY_EXTRACTED

### Capability unavailable

    status = CAPABILITY_UNAVAILABLE

### Requires escalation

    status = ESCALATION_REQUIRED

CEREBRO must not silently replace failed extraction with fabricated content.

---

## 16. Experimental Method

### Phase A — Load Previous Experiment

Load the persisted artifact-identification and capability-routing records.

Verify:

- artifact SHA-256,
- artifact type,
- modality,
- required capability.

---

### Phase B — Validate Original Artifact

Recalculate SHA-256 from the raw bytes.

The calculated checksum must match the checksum used during identification.

If not:

**STOP — SOURCE INTEGRITY FAILURE**

---

### Phase C — Select Extraction Adapter

Use the capability-routing decision to select the appropriate extraction adapter.

Example:

    PRESENTATION_EXTRACTION
              ↓
        PPTX adapter

The notebook should not choose the adapter directly from the filename.

---

### Phase D — Execute Extraction

Run the selected capability.

Produce one or more derived representations.

---

### Phase E — Preserve Source Locations

Every extracted unit must identify where it originated.

Examples:

    Slide 3
    Page 7
    Paragraph 12
    01:34–01:42
    Bounding box
    Character 120–187

---

### Phase F — Produce Extraction Manifest

Generate a structured manifest describing:

- what was processed,
- how it was processed,
- what was extracted,
- what was not extracted,
- provenance,
- source locations,
- processing status.

---

### Phase G — Integrity Validation

Validate:

- original checksum preserved,
- every representation has provenance,
- every representation has source location where applicable,
- deterministic/model distinction retained,
- no knowledge registration occurred.

---

## 17. Success Criteria

EXP-EXTRACT-001 succeeds when:

1. The artifact is processed using its routed capability.
2. Original bytes remain unchanged.
3. Original SHA-256 remains unchanged.
4. Extraction method is recorded.
5. Tool/model information is recorded where applicable.
6. Extracted representations retain source location.
7. Extracted representations retain source provenance.
8. Partial extraction is explicitly represented.
9. AI usage is explicitly represented.
10. No artifact is registered automatically.
11. No trusted Knowledge Fragment is created.
12. Output can be consumed by Step 10 quality evaluation.

---

## 18. Failure Criteria

The experiment fails if:

- processor is selected solely from filename extension,
- original artifact is modified,
- provenance is lost,
- source location is lost where technically available,
- model-generated content is represented as deterministic extraction,
- partial extraction is reported as complete,
- missing content is fabricated,
- artifact is registered automatically,
- Knowledge Fragments are created prematurely,
- extraction implementation leaks into CEREBRO Core contracts.

---

## 19. Expected Output

The experiment should produce:

    extraction_result
    │
    ├── experiment_id
    ├── source
    │   ├── filename
    │   ├── sha256
    │   ├── artifact_type
    │   └── modality
    │
    ├── routing
    │   ├── capability
    │   ├── implementation
    │   └── method_type
    │
    ├── representations[]
    │   ├── representation_id
    │   ├── representation_type
    │   ├── content
    │   ├── source_location
    │   ├── extraction_method
    │   └── provenance
    │
    ├── extraction_summary
    │
    ├── integrity
    │
    └── status

---

## 20. Output Persistence

Persist the experimental result independently from the trusted CEREBRO knowledge store.

Suggested location:

    poc/outputs/experiments/
        EXP-EXTRACT-001.json

Large derived artifacts should be stored separately where appropriate.

Example:

    poc/data/derived/
        <source-sha256>/
            text/
            images/
            audio/
            frames/

The experiment manifest should reference these representations rather than embedding large binary payloads directly.

---

## 21. Relationship to CEREBRO Integrity Framework

Step 09 extends the CEREBRO integrity chain:

**Original Artifact**
↓
**Identification**
↓
**Capability Routing**
↓
**Extraction**
↓
**Derived Representation**

Later:

**Derived Representation**
↓
Quality Evaluation
↓
Normalization
↓
Human Review
↓
Registration
↓
Knowledge Construction
↓
Retrieval
↓
Assisted Recollection
↓
Original Evidence

The original artifact remains the root of provenance.

---

## 22. Expected Architectural Learning

Step 09 should help answer whether CEREBRO can maintain a common ingestion architecture while allowing different modalities to use different processing capabilities.

The desired result is not:

    TXT pipeline
    PDF pipeline
    PPTX pipeline
    Audio pipeline
    Video pipeline

as unrelated systems.

Instead:

                    ARTIFACT
                       │
                       ▼
                   IDENTIFY
                       │
                       ▼
                    ROUTE
                       │
          ┌────────────┼────────────┐
          ▼            ▼            ▼
     deterministic   local AI    specialist
          │            │            │
          └────────────┼────────────┘
                       ▼
             DERIVED REPRESENTATION
                       │
                       ▼
                  EVALUATE
                       │
                       ▼
                  NORMALIZE

The modality-specific implementation should remain behind the extraction capability boundary.

---

## 23. Experiment Principle

> **Different artifacts require different ways of being understood, but they should enter the Digital Knowledge Twin through the same integrity framework.**

Step 09 therefore tests CEREBRO's ability to move from:

**“I know what this artifact is.”**

to:

**“I can extract what it contains while preserving exactly where that information came from.”**

### 1 - Load & Validate Step 07 Handoff

In [2]:
# ---------------------------------------------------------
# CEREBRO STEP 08
# EXP-ROUTE-001 — Adaptive Processing Route Selection
#
# Cell 1 — Load and Validate Step 07 Handoff
# ---------------------------------------------------------

from pathlib import Path
import json


# ---------------------------------------------------------
# Resolve repository root
# ---------------------------------------------------------

repo_root = Path.cwd().parents[1]

step07_path = (
    repo_root
    / "poc"
    / "outputs"
    / "experiments"
    / "EXP-IDENTIFY-001.json"
)

assert step07_path.exists(), (
    f"Step 07 result not found: {step07_path}"
)


# ---------------------------------------------------------
# Load persisted Step 07 result
# ---------------------------------------------------------

with open(
    step07_path,
    "r",
    encoding="utf-8"
) as f:

    step07_result = json.load(f)


# ---------------------------------------------------------
# Validate experiment identity
# ---------------------------------------------------------

assert (
    step07_result.get("experiment_id")
    == "EXP-IDENTIFY-001"
), "Unexpected Step 07 experiment."


# ---------------------------------------------------------
# Validate minimum artifact contract
# ---------------------------------------------------------

artifact = step07_result.get("artifact", {})

required_artifact_fields = [
    "filename",
    "sha256",
    "artifact_type",
    "modality",
]

missing_fields = [
    field
    for field in required_artifact_fields
    if not artifact.get(field)
]

assert not missing_fields, (
    f"Step 07 handoff missing fields: {missing_fields}"
)


# ---------------------------------------------------------
# Validate identification
# ---------------------------------------------------------

identification = step07_result.get(
    "identification",
    {}
)

assert identification.get("status") == "IDENTIFIED", (
    "Step 08 cannot route an artifact that has not "
    "been successfully identified."
)


# ---------------------------------------------------------
# Integrity boundary
# ---------------------------------------------------------

integrity = step07_result.get("integrity", {})

assert (
    integrity.get("artifact_registered") is False
), "Artifact must not already be registered."

assert (
    integrity.get("knowledge_constructed") is False
), "Knowledge must not already be constructed."


# ---------------------------------------------------------
# Establish Step 08 input
# ---------------------------------------------------------

routing_input = {
    "filename":
        artifact["filename"],

    "sha256":
        artifact["sha256"],

    "artifact_type":
        artifact["artifact_type"],

    "modality":
        artifact["modality"],

    "container":
        artifact.get("container"),

    "identification_method":
        identification.get("method"),

    "filename_consistency":
        identification.get(
            "filename_consistency"
        ),
}


# ---------------------------------------------------------
# Display handoff
# ---------------------------------------------------------

print("CEREBRO — STEP 08")
print("Adaptive Processing Route Selection")
print("=" * 65)

print(f"Source Experiment : {step07_result['experiment_id']}")
print(f"Filename          : {routing_input['filename']}")
print(f"SHA-256           : {routing_input['sha256']}")
print(f"Artifact Type     : {routing_input['artifact_type']}")
print(f"Modality          : {routing_input['modality']}")
print(f"Container         : {routing_input['container']}")

print()
print("Step 07 Handoff   : VALID")
print("Artifact Registered: False")
print("Knowledge Created : False")

print()
print("STEP 08 QUESTION:")
print(
    "Given this identified artifact, "
    "how should CEREBRO process it?"
)

CEREBRO — STEP 08
Adaptive Processing Route Selection
Source Experiment : EXP-IDENTIFY-001
Filename          : D5_Assignment.zip
SHA-256           : f3143d4eaedb927eee64ade1eb3d4f0794c6d0323c42ef750d08d7087f083aaf
Artifact Type     : ZIP
Modality          : archive
Container         : ZIP

Step 07 Handoff   : VALID
Artifact Registered: False
Knowledge Created : False

STEP 08 QUESTION:
Given this identified artifact, how should CEREBRO process it?


### 2 - Resolve Required Processing Capability

In [7]:
# ---------------------------------------------------------
# CEREBRO STEP 08
# EXP-ROUTE-001 — Adaptive Processing Route Selection
#
# Cell 2 — Artifact → Capability Routing
# ---------------------------------------------------------


# ---------------------------------------------------------
# Canonical CEREBRO capability contract
# ---------------------------------------------------------

CAPABILITY_MAP = {

    # Text
    "TXT": "TEXT_EXTRACTION",

    # Documents
    "PDF": "PDF_EXTRACTION",
    "DOCX": "DOCUMENT_EXTRACTION",

    # Presentation
    "PPTX": "PRESENTATION_EXTRACTION",

    # Spreadsheet
    "XLSX": "SPREADSHEET_EXTRACTION",

    # Images
    "PNG": "IMAGE_EXTRACTION",
    "JPEG": "IMAGE_EXTRACTION",
    "JPG": "IMAGE_EXTRACTION",

    # Audio
    "WAV": "AUDIO_TRANSCRIPTION",
    "MP3": "AUDIO_TRANSCRIPTION",

    # Video
    "MP4": "VIDEO_EXTRACTION",
}


# ---------------------------------------------------------
# Normalize artifact type
#
# Important:
# Normalize vocabulary only.
# Do NOT reinterpret the artifact.
# ---------------------------------------------------------

raw_artifact_type = routing_input["artifact_type"]

canonical_artifact_type = (
    str(raw_artifact_type)
    .strip()
    .upper()
)


# ---------------------------------------------------------
# Resolve capability
# ---------------------------------------------------------

required_capability = CAPABILITY_MAP.get(
    canonical_artifact_type
)


if required_capability is None:

    routing_status = "UNRESOLVED"

else:

    routing_status = "CAPABILITY_RESOLVED"


# ---------------------------------------------------------
# Build routing decision
# ---------------------------------------------------------

capability_route = {

    "artifact_type_original":
        raw_artifact_type,

    "artifact_type_canonical":
        canonical_artifact_type,

    "modality":
        routing_input["modality"],

    "required_capability":
        required_capability,

    "routing_method":
        "deterministic_capability_mapping",

    "ai_used":
        False,

    "status":
        routing_status,
}


# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

print("CEREBRO — CAPABILITY ROUTING")
print("=" * 65)

print(
    f"Original Artifact Type : "
    f"{capability_route['artifact_type_original']}"
)

print(
    f"Canonical Artifact Type: "
    f"{capability_route['artifact_type_canonical']}"
)

print(
    f"Modality               : "
    f"{capability_route['modality']}"
)

print(
    f"Required Capability    : "
    f"{capability_route['required_capability']}"
)

print(
    f"Routing Method         : "
    f"{capability_route['routing_method']}"
)

print(
    f"AI Used                : "
    f"{capability_route['ai_used']}"
)

print(
    f"Status                 : "
    f"{capability_route['status']}"
)


# ---------------------------------------------------------
# Step 08 must not silently continue if unsupported
# ---------------------------------------------------------

if routing_status == "UNRESOLVED":

    print()
    print("⚠ ROUTING GAP DETECTED")
    print(
        "No canonical processing capability exists "
        f"for artifact type: {canonical_artifact_type}"
    )

else:

    print()
    print(
        "✓ Artifact type successfully mapped "
        "to a CEREBRO processing capability."
    )

CEREBRO — CAPABILITY ROUTING
Original Artifact Type : ZIP
Canonical Artifact Type: ZIP
Modality               : archive
Required Capability    : None
Routing Method         : deterministic_capability_mapping
AI Used                : False
Status                 : UNRESOLVED

⚠ ROUTING GAP DETECTED
No canonical processing capability exists for artifact type: ZIP


### 2A -  Diagnostic Cell 2A

In [8]:
# ---------------------------------------------------------
# CEREBRO STEP 08
# Diagnostic Cell 2A
# Why is capability routing unresolved?
# ---------------------------------------------------------

print("CEREBRO — ROUTING DIAGNOSTIC")
print("=" * 65)

print(
    "Raw Artifact Type       :",
    repr(routing_input.get("artifact_type"))
)

print(
    "Canonical Artifact Type :",
    repr(canonical_artifact_type)
)

print(
    "Modality                :",
    repr(routing_input.get("modality"))
)

print(
    "Container               :",
    repr(routing_input.get("container"))
)

print(
    "Required Capability     :",
    repr(required_capability)
)

print(
    "Routing Status          :",
    repr(capability_route["status"])
)

print()
print("Currently Supported Artifact Types")
print("-" * 65)

for artifact_type, capability in CAPABILITY_MAP.items():
    print(
        f"{artifact_type:10} → {capability}"
    )

CEREBRO — ROUTING DIAGNOSTIC
Raw Artifact Type       : 'ZIP'
Canonical Artifact Type : 'ZIP'
Modality                : 'archive'
Container               : 'ZIP'
Required Capability     : None
Routing Status          : 'UNRESOLVED'

Currently Supported Artifact Types
-----------------------------------------------------------------
TXT        → TEXT_EXTRACTION
PDF        → PDF_EXTRACTION
DOCX       → DOCUMENT_EXTRACTION
PPTX       → PRESENTATION_EXTRACTION
XLSX       → SPREADSHEET_EXTRACTION
PNG        → IMAGE_EXTRACTION
JPEG       → IMAGE_EXTRACTION
JPG        → IMAGE_EXTRACTION
WAV        → AUDIO_TRANSCRIPTION
MP3        → AUDIO_TRANSCRIPTION
MP4        → VIDEO_EXTRACTION


### 2B - Resolve Capability Routing Gap

In [10]:
# ---------------------------------------------------------
# CEREBRO STEP 08
# EXP-ROUTE-001
#
# Cell 2B — Resolve Capability Routing Gap
# ---------------------------------------------------------


# ---------------------------------------------------------
# Canonical vocabulary aliases
#
# These normalize equivalent identification labels.
# They do NOT infer a new artifact type.
# ---------------------------------------------------------

ARTIFACT_TYPE_ALIASES = {

    # Text
    "TEXT": "TXT",
    "PLAIN_TEXT": "TXT",

    # PDF
    "APPLICATION/PDF": "PDF",

    # Word / OOXML
    "WORD": "DOCX",
    "WORD_DOCUMENT": "DOCX",
    "MICROSOFT_WORD": "DOCX",

    # PowerPoint / OOXML
    "POWERPOINT": "PPTX",
    "POWERPOINT_PRESENTATION": "PPTX",
    "PRESENTATION": "PPTX",

    # Excel / OOXML
    "EXCEL": "XLSX",
    "EXCEL_WORKBOOK": "XLSX",
    "WORKBOOK": "XLSX",

    # Images
    "JPG": "JPEG",
    "IMAGE/JPEG": "JPEG",
    "IMAGE/PNG": "PNG",

    # Audio
    "AUDIO/WAV": "WAV",
    "AUDIO/MPEG": "MP3",

    # Video
    "VIDEO/MP4": "MP4",
}


# ---------------------------------------------------------
# Modality fallback
#
# IMPORTANT:
# Used only when Step 07 already identified a meaningful
# modality but artifact-type vocabulary was unresolved.
#
# This selects a CAPABILITY, not a new artifact type.
# ---------------------------------------------------------

MODALITY_CAPABILITY_MAP = {

    "text":
        "TEXT_EXTRACTION",

    "document":
        "DOCUMENT_EXTRACTION",

    "presentation":
        "PRESENTATION_EXTRACTION",

    "spreadsheet":
        "SPREADSHEET_EXTRACTION",

    "image":
        "IMAGE_EXTRACTION",

    "audio":
        "AUDIO_TRANSCRIPTION",

    "video":
        "VIDEO_EXTRACTION",
}


# ---------------------------------------------------------
# Normalize existing Step 07 evidence
# ---------------------------------------------------------

raw_artifact_type = str(
    routing_input.get("artifact_type") or ""
).strip()

raw_modality = str(
    routing_input.get("modality") or ""
).strip()


artifact_key = raw_artifact_type.upper()

modality_key = raw_modality.lower()


# ---------------------------------------------------------
# Stage 1 — Exact canonical artifact mapping
# ---------------------------------------------------------

canonical_artifact_type = artifact_key

required_capability = CAPABILITY_MAP.get(
    canonical_artifact_type
)

resolution_method = None


if required_capability is not None:

    resolution_method = (
        "canonical_artifact_type"
    )


# ---------------------------------------------------------
# Stage 2 — Artifact vocabulary alias
# ---------------------------------------------------------

if required_capability is None:

    alias = ARTIFACT_TYPE_ALIASES.get(
        artifact_key
    )

    if alias is not None:

        canonical_artifact_type = alias

        required_capability = (
            CAPABILITY_MAP.get(alias)
        )

        if required_capability is not None:

            resolution_method = (
                "artifact_type_alias"
            )


# ---------------------------------------------------------
# Stage 3 — Identified modality fallback
#
# We retain the original artifact type.
# We do NOT pretend that modality determines file format.
# ---------------------------------------------------------

if required_capability is None:

    modality_capability = (
        MODALITY_CAPABILITY_MAP.get(
            modality_key
        )
    )

    if modality_capability is not None:

        required_capability = (
            modality_capability
        )

        resolution_method = (
            "identified_modality_fallback"
        )


# ---------------------------------------------------------
# Final routing state
# ---------------------------------------------------------

if required_capability is None:

    routing_status = "UNRESOLVED"

else:

    routing_status = "CAPABILITY_RESOLVED"


# ---------------------------------------------------------
# Rebuild capability route
# ---------------------------------------------------------

capability_route = {

    "artifact_type_original":
        raw_artifact_type,

    "artifact_type_canonical":
        canonical_artifact_type,

    "modality":
        raw_modality,

    "container":
        routing_input.get("container"),

    "required_capability":
        required_capability,

    "resolution_method":
        resolution_method,

    "routing_method":
        "deterministic_hierarchical_routing",

    "ai_used":
        False,

    "status":
        routing_status,
}


# ---------------------------------------------------------
# Display result
# ---------------------------------------------------------

print("CEREBRO — CAPABILITY ROUTING RESOLUTION")
print("=" * 65)

print(
    f"Original Artifact Type : "
    f"{capability_route['artifact_type_original']}"
)

print(
    f"Canonical Artifact Type: "
    f"{capability_route['artifact_type_canonical']}"
)

print(
    f"Modality               : "
    f"{capability_route['modality']}"
)

print(
    f"Container              : "
    f"{capability_route['container']}"
)

print(
    f"Required Capability    : "
    f"{capability_route['required_capability']}"
)

print(
    f"Resolution Method      : "
    f"{capability_route['resolution_method']}"
)

print(
    f"AI Used                : "
    f"{capability_route['ai_used']}"
)

print(
    f"Status                 : "
    f"{capability_route['status']}"
)


# ---------------------------------------------------------
# Guardrail
# ---------------------------------------------------------

assert capability_route["status"] == "CAPABILITY_RESOLVED", (
    "Capability remains unresolved after deterministic "
    "artifact-type, alias, and modality routing. "
    "Do not continue to extraction."
)


print()
print("✓ Processing capability resolved.")
print("✓ No extraction performed.")
print("✓ No model selected.")
print("✓ No AI used.")

CEREBRO — CAPABILITY ROUTING RESOLUTION
Original Artifact Type : ZIP
Canonical Artifact Type: ZIP
Modality               : archive
Container              : ZIP
Required Capability    : None
Resolution Method      : None
AI Used                : False
Status                 : UNRESOLVED


AssertionError: Capability remains unresolved after deterministic artifact-type, alias, and modality routing. Do not continue to extraction.

### 2C - Inspect Raw

In [11]:
# ---------------------------------------------------------
# CEREBRO STEP 08
# EXP-ROUTE-001
#
# Cell 2C — Inspect Raw Step 07 Identification
# ---------------------------------------------------------

print("CEREBRO — RAW STEP 07 HANDOFF")
print("=" * 65)

print("\nARTIFACT")
print("-" * 65)
print(json.dumps(
    step07_result.get("artifact", {}),
    indent=2
))

print("\nIDENTIFICATION")
print("-" * 65)
print(json.dumps(
    step07_result.get("identification", {}),
    indent=2
))

print("\nROUTING INPUT")
print("-" * 65)
print(json.dumps(
    routing_input,
    indent=2
))

print("\nVALUES REQUIRING RESOLUTION")
print("-" * 65)

print(
    "artifact_type =",
    repr(routing_input.get("artifact_type"))
)

print(
    "modality      =",
    repr(routing_input.get("modality"))
)

print(
    "container     =",
    repr(routing_input.get("container"))
)

print(
    "filename      =",
    repr(routing_input.get("filename"))
)

print()
print("✓ Diagnostic complete.")
print("No routing decision changed.")
print("No extraction performed.")

CEREBRO — RAW STEP 07 HANDOFF

ARTIFACT
-----------------------------------------------------------------
{
  "filename": "D5_Assignment.zip",
  "sha256": "f3143d4eaedb927eee64ade1eb3d4f0794c6d0323c42ef750d08d7087f083aaf",
  "artifact_type": "ZIP",
  "modality": "archive",
  "container": "ZIP"
}

IDENTIFICATION
-----------------------------------------------------------------
{
  "status": "IDENTIFIED",
  "method": "deterministic_container_inspection",
  "evidence": [],
  "filename_consistency": "MATCH"
}

ROUTING INPUT
-----------------------------------------------------------------
{
  "filename": "D5_Assignment.zip",
  "sha256": "f3143d4eaedb927eee64ade1eb3d4f0794c6d0323c42ef750d08d7087f083aaf",
  "artifact_type": "ZIP",
  "modality": "archive",
  "container": "ZIP",
  "identification_method": "deterministic_container_inspection",
  "filename_consistency": "MATCH"
}

VALUES REQUIRING RESOLUTION
-----------------------------------------------------------------
artifact_type = 'ZIP'


### 2D - Route Container for Member Inspection

In [16]:
# ---------------------------------------------------------
# CEREBRO STEP 08
# EXP-ROUTE-001
#
# Cell 2D — Container Routing
# ---------------------------------------------------------

artifact_type = (
    str(routing_input.get("artifact_type") or "")
    .strip()
    .upper()
)

modality = (
    str(routing_input.get("modality") or "")
    .strip()
    .lower()
)

container = (
    str(routing_input.get("container") or "")
    .strip()
    .upper()
)


# ---------------------------------------------------------
# Container capability contract
# ---------------------------------------------------------

CONTAINER_CAPABILITY_MAP = {

    "ZIP": "ARCHIVE_MEMBER_INSPECTION",

}


# ---------------------------------------------------------
# Determine whether this is a container artifact
# ---------------------------------------------------------

is_container = (
    modality == "archive"
    or container in CONTAINER_CAPABILITY_MAP
)


assert is_container, (
    "Cell 2D is only for container/archive artifacts."
)


# ---------------------------------------------------------
# Resolve container processing capability
# ---------------------------------------------------------

container_capability = (
    CONTAINER_CAPABILITY_MAP.get(container)
)

assert container_capability is not None, (
    f"No container processing capability defined for: "
    f"{container}"
)


# ---------------------------------------------------------
# Build routing decision
# ---------------------------------------------------------

capability_route = {

    "artifact_type_original":
        routing_input["artifact_type"],

    "artifact_type_canonical":
        artifact_type,

    "modality":
        modality,

    "container":
        container,

    "required_capability":
        container_capability,

    "processing_scope":
        "CONTAINER",

    "member_routing_required":
        True,

    "routing_method":
        "deterministic_container_routing",

    "ai_used":
        False,

    "status":
        "CAPABILITY_RESOLVED",
}


# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

print("CEREBRO — CONTAINER ROUTING")
print("=" * 65)

print(
    f"Artifact Type          : "
    f"{capability_route['artifact_type_canonical']}"
)

print(
    f"Modality               : "
    f"{capability_route['modality']}"
)

print(
    f"Container              : "
    f"{capability_route['container']}"
)

print(
    f"Required Capability    : "
    f"{capability_route['required_capability']}"
)

print(
    f"Processing Scope       : "
    f"{capability_route['processing_scope']}"
)

print(
    f"Member Routing Required: "
    f"{capability_route['member_routing_required']}"
)

print(
    f"AI Used                : "
    f"{capability_route['ai_used']}"
)

print(
    f"Status                 : "
    f"{capability_route['status']}"
)

print()
print("✓ ZIP recognized as a container.")
print("✓ Container routing resolved.")
print("✓ Members must be inspected before extraction.")
print("✓ No archive member extracted yet.")
print("✓ No AI used.")

CEREBRO — CONTAINER ROUTING
Artifact Type          : ZIP
Modality               : archive
Container              : ZIP
Required Capability    : ARCHIVE_MEMBER_INSPECTION
Processing Scope       : CONTAINER
Member Routing Required: True
AI Used                : False
Status                 : CAPABILITY_RESOLVED

✓ ZIP recognized as a container.
✓ Container routing resolved.
✓ Members must be inspected before extraction.
✓ No archive member extracted yet.
✓ No AI used.


### 3 - Decompose Capability into Processing Requirements

In [18]:
# ---------------------------------------------------------
# CEREBRO STEP 08
# EXP-ROUTE-001 — Adaptive Processing Route Selection
#
# Cell 3 — Container Processing Requirements
# ---------------------------------------------------------


# ---------------------------------------------------------
# Preconditions
# ---------------------------------------------------------

assert capability_route["status"] == "CAPABILITY_RESOLVED", (
    "Processing capability has not been resolved."
)

required_capability = capability_route[
    "required_capability"
]


# ---------------------------------------------------------
# Container branch
# ---------------------------------------------------------

if required_capability == "ARCHIVE_MEMBER_INSPECTION":

    processing_requirements = {

        "artifact_type":
            capability_route[
                "artifact_type_canonical"
            ],

        "capability":
            required_capability,

        "processing_scope":
            "CONTAINER",

        "requirements": {

            # Inspect archive structure
            "manifest_inspection": True,

            # Identify individual contained files
            "member_identification": True,

            # Preserve archive hierarchy/path
            "hierarchy_preservation": True,

            # Each member will later receive
            # its own processing route
            "member_routing": True,

            # Do NOT extract knowledge yet
            "content_extraction": False,

            # Do NOT invoke AI yet
            "ai_processing": False,
        },

        "processing_complexity":
            "MULTI_ARTIFACT_CONTAINER",

        "implementation_selected":
            False,

        "model_selected":
            False,

        "ai_used":
            False,

        "status":
            "REQUIREMENTS_DEFINED",
    }


# ---------------------------------------------------------
# Non-container branch
#
# Retain existing processing requirement logic for
# ordinary artifacts.
# ---------------------------------------------------------

else:

    requirements = PROCESSING_REQUIREMENTS.get(
        required_capability
    )

    assert requirements is not None, (
        f"No processing requirements defined for "
        f"capability: {required_capability}"
    )

    required_components = [
        component
        for component, required
        in requirements.items()
        if required
    ]

    processing_complexity = (
        "SINGLE_CAPABILITY"
        if len(required_components) == 1
        else "MULTI_CAPABILITY"
    )

    processing_requirements = {

        "artifact_type":
            capability_route[
                "artifact_type_canonical"
            ],

        "capability":
            required_capability,

        "processing_scope":
            "ARTIFACT",

        "requirements":
            requirements,

        "required_components":
            required_components,

        "processing_complexity":
            processing_complexity,

        "implementation_selected":
            False,

        "model_selected":
            False,

        "ai_used":
            False,

        "status":
            "REQUIREMENTS_DEFINED",
    }


# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

print("CEREBRO — PROCESSING REQUIREMENTS")
print("=" * 65)

print(
    f"Artifact Type : "
    f"{processing_requirements['artifact_type']}"
)

print(
    f"Capability    : "
    f"{processing_requirements['capability']}"
)

print(
    f"Scope         : "
    f"{processing_requirements['processing_scope']}"
)

print(
    f"Complexity    : "
    f"{processing_requirements['processing_complexity']}"
)

print()
print("Requirements")
print("-" * 65)

for name, required in (
    processing_requirements["requirements"].items()
):

    marker = "✓" if required else "—"

    print(
        f"{marker} {name:25} "
        f"{'REQUIRED' if required else 'NOT REQUIRED'}"
    )


print()
print(
    "Implementation Selected :",
    processing_requirements[
        "implementation_selected"
    ]
)

print(
    "Model Selected          :",
    processing_requirements[
        "model_selected"
    ]
)

print(
    "AI Used                 :",
    processing_requirements[
        "ai_used"
    ]
)

print()
print(
    f"Status: "
    f"{processing_requirements['status']}"
)

CEREBRO — PROCESSING REQUIREMENTS
Artifact Type : ZIP
Capability    : ARCHIVE_MEMBER_INSPECTION
Scope         : CONTAINER
Complexity    : MULTI_ARTIFACT_CONTAINER

Requirements
-----------------------------------------------------------------
✓ manifest_inspection       REQUIRED
✓ member_identification     REQUIRED
✓ hierarchy_preservation    REQUIRED
✓ member_routing            REQUIRED
— content_extraction        NOT REQUIRED
— ai_processing             NOT REQUIRED

Implementation Selected : False
Model Selected          : False
AI Used                 : False

Status: REQUIREMENTS_DEFINED


### 3A — Persist Original Artifact

In [20]:
# ---------------------------------------------------------
# CEREBRO STEP 08
# Repair Cell 3A
#
# Persist original uploaded artifact before processing
# ---------------------------------------------------------

from pathlib import Path
import hashlib


# ---------------------------------------------------------
# Verify original bytes are still available
# ---------------------------------------------------------

assert "unknown_bytes" in globals(), (
    "Original upload bytes are no longer available in "
    "the notebook kernel. Re-upload D5_Assignment.zip."
)

assert unknown_bytes, (
    "unknown_bytes is empty."
)


# ---------------------------------------------------------
# Verify against Step 07 SHA-256
# ---------------------------------------------------------

expected_sha256 = routing_input["sha256"]

actual_sha256 = hashlib.sha256(
    unknown_bytes
).hexdigest()


assert actual_sha256 == expected_sha256, (
    "SOURCE INTEGRITY FAILURE: current bytes do not "
    "match the artifact identified in Step 07."
)


# ---------------------------------------------------------
# Persist using SHA-based artifact directory
#
# This prevents filename collisions and preserves
# source identity.
# ---------------------------------------------------------

artifact_dir = (
    repo_root
    / "poc"
    / "data"
    / "raw"
    / "unknown"
    / expected_sha256
)

artifact_dir.mkdir(
    parents=True,
    exist_ok=True
)


source_path = (
    artifact_dir
    / routing_input["filename"]
)


# ---------------------------------------------------------
# Write original bytes
# ---------------------------------------------------------

source_path.write_bytes(
    unknown_bytes
)


# ---------------------------------------------------------
# Re-read and independently verify persisted artifact
# ---------------------------------------------------------

persisted_bytes = source_path.read_bytes()

persisted_sha256 = hashlib.sha256(
    persisted_bytes
).hexdigest()


assert persisted_sha256 == expected_sha256, (
    "Persisted artifact failed SHA-256 verification."
)


# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

print("CEREBRO — ORIGINAL ARTIFACT PERSISTENCE")
print("=" * 70)

print(
    f"Filename : "
    f"{routing_input['filename']}"
)

print(
    f"Path     : "
    f"{source_path.relative_to(repo_root)}"
)

print(
    f"SHA-256  : "
    f"{persisted_sha256}"
)

print(
    f"Size     : "
    f"{len(persisted_bytes):,} bytes"
)

print()
print("✓ Original artifact persisted.")
print("✓ SHA-256 verified after persistence.")
print("✓ Original filename preserved.")
print("✓ No extraction performed.")
print("✓ No AI used.")
print("✓ Artifact NOT registered.")
print("✓ Knowledge NOT constructed.")

AssertionError: Original upload bytes are no longer available in the notebook kernel. Re-upload D5_Assignment.zip.

### 4 - Inspect ZIP Manifest

In [19]:
# ---------------------------------------------------------
# CEREBRO STEP 08
# EXP-ROUTE-001 — Adaptive Processing Route Selection
#
# Cell 4 — Inspect ZIP Member Manifest
# ---------------------------------------------------------

import zipfile
from pathlib import Path


# ---------------------------------------------------------
# Preconditions
# ---------------------------------------------------------

assert (
    processing_requirements["status"]
    == "REQUIREMENTS_DEFINED"
)

assert (
    processing_requirements["capability"]
    == "ARCHIVE_MEMBER_INSPECTION"
)

assert (
    processing_requirements["processing_scope"]
    == "CONTAINER"
)


# ---------------------------------------------------------
# Locate original ZIP
#
# Prefer the persisted raw artifact.
# ---------------------------------------------------------

raw_root = repo_root / "poc" / "data" / "raw"

candidate_paths = [
    path
    for path in raw_root.rglob(
        routing_input["filename"]
    )
    if path.is_file()
]


assert candidate_paths, (
    f"Cannot find persisted source artifact: "
    f"{routing_input['filename']}"
)


source_path = candidate_paths[0]


# ---------------------------------------------------------
# Inspect ZIP central directory
#
# No member content processing occurs here.
# ---------------------------------------------------------

archive_members = []


with zipfile.ZipFile(source_path, "r") as archive:

    for info in archive.infolist():

        member_path = Path(info.filename)

        is_directory = info.is_dir()

        archive_members.append({

            "member_path":
                info.filename,

            "filename":
                member_path.name,

            "extension":
                (
                    member_path.suffix.lower()
                    if not is_directory
                    else None
                ),

            "is_directory":
                is_directory,

            "uncompressed_size":
                info.file_size,

            "compressed_size":
                info.compress_size,

            "crc32":
                f"{info.CRC:08x}",
        })


# ---------------------------------------------------------
# Separate files and directories
# ---------------------------------------------------------

file_members = [
    member
    for member in archive_members
    if not member["is_directory"]
]

directory_members = [
    member
    for member in archive_members
    if member["is_directory"]
]


# ---------------------------------------------------------
# Extension inventory
# ---------------------------------------------------------

extension_inventory = {}


for member in file_members:

    extension = (
        member["extension"]
        or "<no extension>"
    )

    extension_inventory[extension] = (
        extension_inventory.get(
            extension,
            0
        ) + 1
    )


# ---------------------------------------------------------
# Build manifest
# ---------------------------------------------------------

container_manifest = {

    "container_filename":
        routing_input["filename"],

    "container_sha256":
        routing_input["sha256"],

    "container_type":
        "ZIP",

    "member_count":
        len(file_members),

    "directory_count":
        len(directory_members),

    "extension_inventory":
        extension_inventory,

    "members":
        archive_members,

    "inspection_method":
        "zip_central_directory",

    "member_content_processed":
        False,

    "ai_used":
        False,

    "status":
        "MANIFEST_INSPECTED",
}


# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

print("CEREBRO — ZIP MANIFEST")
print("=" * 75)

print(
    f"Container   : "
    f"{container_manifest['container_filename']}"
)

print(
    f"Files       : "
    f"{container_manifest['member_count']}"
)

print(
    f"Directories : "
    f"{container_manifest['directory_count']}"
)

print()

print("EXTENSION INVENTORY")
print("-" * 75)

for extension, count in sorted(
    extension_inventory.items()
):

    print(
        f"{extension:15} {count}"
    )


print()
print("ARCHIVE MEMBERS")
print("-" * 75)


for index, member in enumerate(
    file_members,
    start=1
):

    print(
        f"{index:3}. "
        f"{member['member_path']} "
        f"[{member['extension'] or 'no extension'}] "
        f"{member['uncompressed_size']:,} bytes"
    )


print()
print("✓ ZIP manifest inspected.")
print("✓ Archive hierarchy preserved.")
print("✓ Member metadata recorded.")
print("✓ Member content NOT processed.")
print("✓ No extraction performed.")
print("✓ No AI used.")

print()
print("Status: MANIFEST_INSPECTED")

AssertionError: Cannot find persisted source artifact: D5_Assignment.zip